# Milestone 4: QLoRA fine-tuning of Llama-3.1-8B on the competition data

Course: Introduction to DL and GenAI Project
Competition: Smart MCQ Solver Challenge
Roll No: 22f3000114

## Inputs to attach

1. `jjinho/wikipedia-2023-07-faiss-index`
2. `jjinho/wikipedia-20230701`
3. Competition data

GPU T4 x2, secrets HF_TOKEN and WANDB. Full run takes roughly 4 to 6 hours, most of it training. Set EPOCHS to 1 to roughly halve it.

## The idea

This is the Day 8 bootcamp recipe applied to our own competition data. The base Llama-3.1-8B answers from general knowledge. Fine-tuning with LoRA on our 2000 training questions teaches it three specific things: this exact question style, this exact answer format (a single letter after Answer:), and how to use the retrieved Wikipedia context that our inference pipeline provides. Because LoRA only trains small low-rank matrices on top of the frozen 4-bit base (QLoRA), it fits on the T4s.

The training format follows Day 8 exactly: prompt and completion columns, completion_only_loss so the model is only graded on the answer letter, not on reproducing the question. The prompt is character for character the same string our submission notebook builds at inference, including the retrieved context, so there is no train-inference mismatch.

Rows 0 to 199 are held out for validation, they are the same 200 rows the submission notebook validates on, so numbers are comparable. Training uses rows 200 to 1999.

At the end the LoRA adapter (a few tens of MB) is saved to /kaggle/working. Publish it as a Kaggle dataset (File, Create dataset from output) and attach it to the submission notebook, which will then ensemble the tuned and base model automatically.


## Install, imports and settings


In [1]:
!pip install -qU transformers accelerate bitsandbytes peft trl datasets sentence-transformers faiss-cpu wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 78.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 56.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 24.3 MB/s eta 0:00:00


In [2]:
import glob, os, gc, pickle, numpy as np, pandas as pd, torch
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.auto import tqdm
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print('CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())

OPTIONS = ['A', 'B', 'C', 'D', 'E']

# retrieval settings, smaller than the submission notebook to keep this run shorter
K_ARTICLES = 5      # articles per option query (submission uses 10)
K_STAGE1   = 20
K_FINAL    = 3      # chunks in the training context (submission uses 5)
CTX_CHARS  = 1500   # keeps training sequences near 1000 tokens

N_VAL  = 200        # rows 0..199 held out, same split as the submission notebook
EPOCHS = 2          # set to 1 if short on time
MAX_LEN = 896      # token cap for training sequences


CUDA available: True | GPUs: 2


## Load the competition data


In [3]:
def find_file(name):
    for pat in [f'/kaggle/input/**/{name}', f'./{name}', name]:
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(name + ' not found')

train = pd.read_csv(find_file('train.csv'))
print('train:', train.shape)


train: (2000, 8)


## Build retrieved contexts for all 2000 training rows

Same retrieval as the submission notebook: per-option queries into the prebuilt Wikipedia FAISS index, stream the needed articles from parquet with an id-level checkpoint, then two-stage rerank (bi-encoder cut to 20, cross-encoder keep the top 3). The contexts go into the training prompts so the model learns to read retrieved context.


In [4]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

In [5]:
WIKI_INDEX_PATH = glob.glob('/kaggle/input/**/wikipedia_202307.index', recursive=True)
idx_parquet = glob.glob('/kaggle/input/**/wiki_2023_index.parquet', recursive=True)
assert WIKI_INDEX_PATH and idx_parquet, 'Attach both jjinho wikipedia datasets'
WIKI_INDEX_PATH = WIKI_INDEX_PATH[0]
WIKI_PARQUET_DIR = os.path.dirname(idx_parquet[0])

print('Loading FAISS index')
wiki_index = faiss.read_index(WIKI_INDEX_PATH)
wiki_map = pd.read_parquet(idx_parquet[0])

bi_encoder = SentenceTransformer(
    '/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2',
    device='cuda')

def build_queries(row):
    p = str(row['prompt'])
    return [f'{p} {row[o]}' for o in OPTIONS]

all_rows = [train.iloc[i] for i in range(len(train))]
queries = []
for r in all_rows:
    queries.extend(build_queries(r))

q_emb = bi_encoder.encode(queries, batch_size=64, show_progress_bar=True)
q_emb = np.asarray(q_emb, dtype='float32')
_, art = wiki_index.search(q_emb, K_ARTICLES)
article_rows = art.reshape(len(all_rows), 5 * K_ARTICLES)
print('article_rows:', article_rows.shape)


Loading FAISS index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

article_rows: (2000, 25)


In [6]:
import pyarrow.parquet as pq

def save_ckpt(name, obj):
    with open(f'/kaggle/working/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)

def load_ckpt(name):
    p = f'/kaggle/working/{name}.pkl'
    return pickle.load(open(p, 'rb')) if os.path.exists(p) else None

def chunk_article(title, text):
    parts = [p.strip() for p in str(text).split('\n') if len(p.strip()) > 80]
    if not parts:
        t = str(text)
        parts = [t[i:i+800] for i in range(0, min(len(t), 30000), 700)]
    return [f'{title}. {p}'[:1500] for p in parts]

needed = wiki_map.iloc[article_rows.flatten()].copy()
file_col = 'file' if 'file' in needed.columns else needed.columns[-1]
id_col   = 'id'   if 'id'   in needed.columns else needed.columns[0]
needed_by_file = needed.groupby(file_col)[id_col].apply(
    lambda s: set(str(x) for x in s)).to_dict()

article_chunks = load_ckpt('article_chunks_m4') or {}
for fname, id_set in tqdm(needed_by_file.items(), desc='parquet files'):
    missing = {i for i in id_set if i not in article_chunks}
    if not missing:
        continue
    fpath = os.path.join(WIKI_PARQUET_DIR, str(fname))
    if not os.path.exists(fpath):
        hits = glob.glob(os.path.join(WIKI_PARQUET_DIR, '**', str(fname)), recursive=True)
        if not hits:
            continue
        fpath = hits[0]
    pf = pq.ParquetFile(fpath)
    tcol = 'text' if 'text' in pf.schema.names else [c for c in pf.schema.names if c not in ('id', 'title')][-1]
    cols = [c for c in ['id', 'title', tcol] if c in pf.schema.names]
    for batch in pf.iter_batches(batch_size=5000, columns=cols):
        bdf = batch.to_pandas()
        bdf['id'] = bdf['id'].astype(str)
        hit = bdf[bdf['id'].isin(missing)]
        for _, r in hit.iterrows():
            title = str(r['title']) if 'title' in bdf.columns else ''
            article_chunks[r['id']] = chunk_article(title, r[tcol])
        del bdf, hit
    del pf
    gc.collect()
    save_ckpt('article_chunks_m4', article_chunks)

all_wanted = set(str(x) for x in needed[id_col])
cov = len(all_wanted & set(article_chunks))
print(f'coverage: {cov} of {len(all_wanted)} ({100*cov/len(all_wanted):.1f} percent)')
assert cov / len(all_wanted) > 0.95, 'coverage below 95 percent, investigate before training'


parquet files:   0%|          | 0/28 [00:00<?, ?it/s]

coverage: 2719 of 2719 (100.0 percent)


In [7]:
contexts = load_ckpt('contexts_m4')
if contexts is None:
    contexts = []
    for qi in tqdm(range(len(all_rows)), desc='retrieve (bi-encoder only)'):
        row = all_rows[qi]
        seen, chunks = set(), []
        for art_row in article_rows[qi]:
            if art_row in seen:
                continue
            seen.add(art_row)
            aid = str(wiki_map.iloc[art_row][id_col])
            chunks.extend(article_chunks.get(aid, []))
        if not chunks:
            contexts.append(''); continue
    
        q_text = str(row['prompt']) + ' ' + ' '.join(str(row[o]) for o in OPTIONS)
        c_emb = bi_encoder.encode(chunks, batch_size=128, show_progress_bar=False)
        qv = bi_encoder.encode([q_text], show_progress_bar=False)[0]
        sims = c_emb @ qv / (np.linalg.norm(c_emb, axis=1) * np.linalg.norm(qv) + 1e-9)
        top = np.argsort(-sims)[:K_FINAL]          # keep the same K_FINAL as before, e.g. 5
        contexts.append(' '.join(chunks[i] for i in top)[:CTX_CHARS])
    save_ckpt('contexts_m4', contexts)

print('contexts ready:', len(contexts), '| empty:', sum(1 for c in contexts if not c))


retrieve (bi-encoder only):   0%|          | 0/2000 [00:00<?, ?it/s]

contexts ready: 2000 | empty: 0


## Free retrieval models before training


In [8]:
del bi_encoder, wiki_index, wiki_map, q_emb, article_chunks
gc.collect()
torch.cuda.empty_cache()
for d in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(d)
    print(f'GPU {d}: {free/1e9:.1f} GB free of {total/1e9:.1f} GB')


GPU 0: 15.5 GB free of 15.6 GB
GPU 1: 15.5 GB free of 15.6 GB


## Format the training data, prompt and completion

Exactly the Day 8 Exercise 6 pattern: a prompt column that ends with Answer: and a completion column that is a leading space plus the answer, here the correct letter. completion_only_loss then masks the prompt, so the loss is computed only on the letter. The prompt string is identical to the one the submission notebook builds at inference.


In [9]:
from datasets import Dataset

def build_prompt_text(row, context):
    q = str(row['prompt'])
    opts = '\n'.join(f'{o}. {row[o]}' for o in OPTIONS)
    if context:
        return f'Context:\n{context}\n\nQuestion: {q}\n\nOptions:\n{opts}\n\nAnswer:'
    return f'Question: {q}\n\nOptions:\n{opts}\n\nAnswer:'

records = []
for i in range(N_VAL, len(train)):          # rows 200..1999, rows 0..199 stay held out
    row = train.iloc[i]
    records.append({
        'prompt': build_prompt_text(row, contexts[i]),
        'completion': f" {row['answer']}",
    })

train_ds = Dataset.from_list(records)
print(train_ds)
print('sample completion:', repr(train_ds[0]['completion']))
print('prompt tail:', repr(train_ds[0]['prompt'][-80:]))


Dataset({
    features: ['prompt', 'completion'],
    num_rows: 1800
})
sample completion: ' C'
prompt tail: 'roup of organisms that do not acquire any energy from the ecostructure.\n\nAnswer:'


## Load the 4-bit base and configure LoRA

QLoRA: the base model is frozen in 4-bit NF4 (float16 compute for the T4s) and LoRA trains small low-rank matrices on the attention projections. r 16, alpha 32, dropout 0.05, targets q, k, v and o projections, the same shape of config as the Day 8 exercises. Trainable parameters are a fraction of a percent of the model.


In [10]:
from peft import LoraConfig, TaskType

model_dir = '/kaggle/input/models/metaresearch/llama-3.1/transformers/8b-instruct/2'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # not bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    quantization_config=bnb_config,
    dtype=torch.float16,              # don't let Llama's bf16 default leak through
    device_map={'': 0},
)
model.config.use_cache = False

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [11]:
'''from kaggle_secrets import UserSecretsClient
from peft import LoraConfig, TaskType

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
model_id = 'meta-llama/Llama-3.1-8B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,   # T4 has no native bfloat16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map='auto',
    low_cpu_mem_usage=True, token=hf_token)
model.config.use_cache = False   # required with gradient checkpointing
'''
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
print('Base loaded. LoRA config ready.')


Base loaded. LoRA config ready.


## Train with SFTTrainer

completion_only_loss masks everything except the answer letter. Batch size 1 with gradient accumulation 16 gives an effective batch of 16 while fitting long context sequences on the T4s. Gradient checkpointing trades compute for memory. Logged to W&B as the Milestone 4 run.


In [12]:
import wandb
try:
    wandb.login(key=UserSecretsClient().get_secret('WANDB'))
    WANDB_OK = True
except Exception as e:
    print('W&B not available:', e); WANDB_OK = False

from trl import SFTTrainer, SFTConfig

sft_args = SFTConfig(
    output_dir='/kaggle/working/m4-adapter',
    completion_only_loss=True,
    loss_type='nll',
    max_length=MAX_LEN,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    save_strategy='no',
    report_to='wandb' if WANDB_OK else 'none',
    run_name='m4-llama3.1-8b-qlora',
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

# fp16 GradScaler can't unscale bf16 grads — put the trainable (LoRA) params in fp32
import torch
n = 0
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float(); n += 1
print('cast', n, 'trainable tensors to fp32')

trainer.train()
print('Training done.')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


cast 256 trainable tensors to fp32


Step,Training Loss
10,7.845640
20,0.441240
30,0.098018
40,0.065160
50,0.091441
60,0.000234
70,0.002300
80,0.000396
90,0.000066
100,0.000049


Training done.


## Save the adapter

Only the adapter is saved, a few tens of MB. It is never merged into the 4-bit base, merging into a quantized model corrupts the weights. The submission notebook loads it on top of the base with PEFT.


In [13]:
trainer.save_model('/kaggle/working/m4-adapter')
print(os.listdir('/kaggle/working/m4-adapter'))
print('Adapter saved. After this notebook finishes, use File -> Create dataset from output,')
print('then attach that dataset as an input to the submission notebook.')


['training_args.bin', 'adapter_config.json', 'tokenizer.json', 'README.md', 'chat_template.jinja', 'tokenizer_config.json', 'adapter_model.safetensors']
Adapter saved. After this notebook finishes, use File -> Create dataset from output,
then attach that dataset as an input to the submission notebook.


## Quick validation on the held-out 200 rows

Logit extraction with the tuned model (adapter active) and the base model (adapter disabled) on the same held-out rows and the same contexts. This is the honest before and after comparison for Milestone 4, on rows the fine-tune never saw.


In [15]:
tuned = trainer.model
tuned.eval()

letter_ids = {}
for o in OPTIONS:
    ids = set()
    ids.add(tokenizer.encode(o, add_special_tokens=False)[-1])
    ids.add(tokenizer.encode(' ' + o, add_special_tokens=False)[-1])
    letter_ids[o] = list(ids)

@torch.no_grad()
def letter_probs(row, context, m):
    input_ids = tokenizer(build_prompt_text(row, context), return_tensors='pt').input_ids
    out = m(input_ids.to(m.device))
    next_logits = out.logits[0, -1]
    raw = torch.tensor([max(next_logits[i].item() for i in letter_ids[o]) for o in OPTIONS])
    return torch.softmax(raw, dim=0).numpy()

def top3(p):
    order = np.argsort(-p)
    return [OPTIONS[i] for i in order[:3]]

def map_at_3(predictions, labels):
    weights = [1.0, 1/2, 1/3]
    total = 0.0
    for preds, true in zip(predictions, labels):
        s = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                s = weights[i]; break
        total += s
    return total / len(labels)

val_true, p_tuned_all, p_base_all = [], [], []
for i in tqdm(range(N_VAL), desc='validate'):
    row = train.iloc[i]
    val_true.append(row['answer'])
    p_tuned_all.append(letter_probs(row, contexts[i], tuned))
    with tuned.disable_adapter():
        p_base_all.append(letter_probs(row, contexts[i], tuned))

m_tuned = map_at_3([top3(p) for p in p_tuned_all], val_true)
m_base  = map_at_3([top3(p) for p in p_base_all], val_true)
m_ens   = map_at_3([top3((a + b) / 2) for a, b in zip(p_tuned_all, p_base_all)], val_true)
print(f'Base model MAP@3 (with context)  : {m_base:.4f}')
print(f'Tuned model MAP@3 (with context) : {m_tuned:.4f}')
print(f'Ensemble MAP@3 (average probs)   : {m_ens:.4f}')

validate:   0%|          | 0/200 [00:00<?, ?it/s]

Base model MAP@3 (with context)  : 0.8908
Tuned model MAP@3 (with context) : 0.9975
Ensemble MAP@3 (average probs)   : 0.9975


In [16]:
# --- UPLOAD M4 ADAPTER TO KAGGLE HUB ---
import kagglehub, shutil, os

adapter_src = '/kaggle/working/m4-adapter'   # where trainer.save_model wrote it

KAGGLE_USERNAME = "dhruviitm"
MODEL_SLUG = "llama31-mcq-lora"
FRAMEWORK = "pytorch"
VARIATION = "m4-qlora-adapter"

upload_dir_adapter = "/kaggle/working/upload_model_m4"
os.makedirs(upload_dir_adapter, exist_ok=True)
for fname in os.listdir(adapter_src):
    shutil.copy(os.path.join(adapter_src, fname), os.path.join(upload_dir_adapter, fname))

MODEL_HANDLE_ADAPTER = f"{KAGGLE_USERNAME}/{MODEL_SLUG}/{FRAMEWORK}/{VARIATION}"

try:
    kagglehub.model_upload(
        MODEL_HANDLE_ADAPTER,
        upload_dir_adapter,
        version_notes="M4 QLoRA adapter, Llama-3.1-8B-Instruct on MCQ train data."
    )
    print("Uploaded adapter:", MODEL_HANDLE_ADAPTER)
except Exception as e:
    print("Upload failed:", e)

Uploading Model https://api.kaggle.com/models/dhruviitm/llama31-mcq-lora/pytorch/m4-qlora-adapter ...
Model 'llama31-mcq-lora' does not exist or access is forbidden for user 'dhruviitm'. Creating or handling Model...
Starting upload for file /kaggle/working/upload_model_m4/training_args.bin


Uploading: 100%|██████████| 5.71k/5.71k [00:00<00:00, 15.8kB/s]

Upload successful: /kaggle/working/upload_model_m4/training_args.bin (6KB)
Starting upload for file /kaggle/working/upload_model_m4/adapter_config.json



Uploading: 100%|██████████| 1.10k/1.10k [00:00<00:00, 3.11kB/s]

Upload successful: /kaggle/working/upload_model_m4/adapter_config.json (1KB)
Starting upload for file /kaggle/working/upload_model_m4/tokenizer.json


...


Uploading: 100%|██████████| 353/353 [00:00<00:00, 950B/s]

Upload successful: /kaggle/working/upload_model_m4/tokenizer_config.json (353B)
Starting upload for file /kaggle/working/upload_model_m4/adapter_model.safetensors



Uploading: 100%|██████████| 54.6M/54.6M [00:02<00:00, 21.8MB/s]

Upload successful: /kaggle/working/upload_model_m4/adapter_model.safetensors (52MB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/dhruviitm/llama31-mcq-lora/pytorch/m4-qlora-adapter
Uploaded adapter: dhruviitm/llama31-mcq-lora/pytorch/m4-qlora-adapter


In [17]:
import kagglehub
adapter_dir = kagglehub.model_download(f"{KAGGLE_USERNAME}/{MODEL_SLUG}/{FRAMEWORK}/{VARIATION}")
print('adapter downloaded to:', adapter_dir)
HAS_ADAPTER = True

adapter downloaded to: /kaggle/input/models/dhruviitm/llama31-mcq-lora/pytorch/m4-qlora-adapter/1


In [18]:
import os
print(os.listdir(adapter_dir))

['adapter_model.safetensors', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']


In [19]:
from peft import PeftModel

model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()
HAS_ADAPTER = True
print('Fine-tuned adapter loaded, running 4-signal ensemble.')

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Fine-tuned adapter loaded, running 4-signal ensemble.


## Notes

Fill in after running.

Record the three validation numbers, base with context, tuned with context, and the ensemble. The tuned model should beat the base, and the ensemble usually edges out both. If the tuned model is worse than the base, the usual causes are too high a learning rate or too many epochs on 1800 rows, retry with 1 epoch.

For the viva, the story of this milestone: LoRA freezes the pretrained base and learns two small low-rank matrices per targeted layer (Day 8 Exercise 2 showed the parameter savings, r times d plus k instead of d times k). QLoRA does that on top of a 4-bit quantized base so an 8B model trains on a 16 GB GPU. completion_only_loss masks the prompt so the model is graded only on the answer letter, the Exercise 6 prompt and completion format. Training prompts include the retrieved Wikipedia context so the model learns to read retrieved evidence, which matches how it is used at inference.
